## Module imports with loading the dataset:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('../data/retail_dataset_cleaned.xlsx')
print("Modules imported successfully and data loaded")

## ABC Analysis

In [ ]:
# cumulative revenue contribution
product_revenue = df.groupby('product')['total_sales'].sum().sort_values(ascending=False).reset_index()
product_revenue['Cumulative_Revenue'] = product_revenue['total_sales'].cumsum()
product_revenue['Cumulative_Percentage'] = (product_revenue['Cumulative_Revenue'] / product_revenue['total_sales'].sum()) * 100

def classify_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

product_revenue['ABC_Category'] = product_revenue['Cumulative_Percentage'].apply(classify_abc)

print("\nABC Analysis:")
print(product_revenue.head(20))
print("\nProduct Distribution:")
print(product_revenue['ABC_Category'].value_counts())

# -- A = Top 20% products contributing 80% revenue
# -- B = Next products contributing 15% revenue
# -- C = Remaining products contributing 5% revenue

plt.figure(figsize=(14, 6))
plt.plot(range(len(product_revenue)), product_revenue['Cumulative_Percentage'], marker='o', markersize=3)
plt.axhline(y=80, color='r', linestyle='--', label='80% Revenue')
plt.axhline(y=95, color='orange', linestyle='--', label='95% Revenue')
plt.fill_between(range(len(product_revenue)), 0, product_revenue['Cumulative_Percentage'], 
                 where=(product_revenue['Cumulative_Percentage'] <= 80), alpha=0.3, color='green', label='Category A')
plt.fill_between(range(len(product_revenue)), 0, product_revenue['Cumulative_Percentage'], 
                 where=((product_revenue['Cumulative_Percentage'] > 80) & (product_revenue['Cumulative_Percentage'] <= 95)), 
                 alpha=0.3, color='yellow', label='Category B')
plt.title('ABC Analysis - Cumulative Revenue by Products', fontsize=14, fontweight='bold')
plt.xlabel('Products (sorted by revenue)')
plt.ylabel('Cumulative Revenue %')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Retailer based analytics:

In [ ]:
retailer_metrics = df.groupby(['retailer', 'retailer_id']).agg({
    'total_sales': 'sum',
    'operating_profit': 'sum',
    'operating_margin': 'mean',  # average profit margin efficiency
    'invoice_date': 'count'      # no. of transaction lines processed
}).rename(columns={'invoice_date': 'transaction_count'})

retailer_metrics['profit_per_transaction'] = retailer_metrics['operating_profit'] / retailer_metrics['transaction_count']

print("\nTop 10 Retailer Accounts by Total Operating Profit Contribution:")
print(retailer_metrics.sort_values('operating_profit', ascending=False).head(10))

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(retailer_metrics['total_sales'], bins=15, color='skyblue', edgecolor='black')
plt.title('Distribution of Total Revenue\nper Retailer Account', fontsize=11, fontweight='bold')
plt.xlabel('Total Sales Value (Rs.)')
plt.ylabel('Count of Retailers')
plt.ticklabel_format(style='plain', axis='x')

plt.subplot(1, 3, 2)
plt.hist(retailer_metrics['transaction_count'], bins=15, color='lightgreen', edgecolor='black')
plt.title('Distribution of Transaction Volume\nper Retailer Account', fontsize=11, fontweight='bold')
plt.xlabel('Number of Line-Item Transactions')
plt.ylabel('Count of Retailers')

plt.subplot(1, 3, 3)
plt.hist(retailer_metrics['operating_margin'] * 100, bins=15, color='salmon', edgecolor='black')
plt.title('Distribution of Avg Margin %\nper Retailer Account', fontsize=11, fontweight='bold')
plt.xlabel('Average Operating Margin (%)')
plt.ylabel('Count of Retailers')

plt.tight_layout()
plt.show()

## Seaborn and NumPy based bar-graph insights:

In [ ]:
conditions = [
    (df['operating_margin'] >= 0.25),
    (df['operating_margin'] >= 0.15) & (df['operating_margin'] < 0.25),
    (df['operating_margin'] < 0.15)
]
choices = ['High Margin Tier', 'Standard Tier', 'Low Margin Tier'] # assigning logic

df['profit_tier'] = np.select(conditions, choices, default='Unclassified') # numpy to assign categories

sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))

sns.barplot(
    data=df, 
    x='region', 
    y='total_sales', 
    hue='sales_method', 
    estimator=sum, # add up total sales per group
    errorbar=None, # cleaner look
    palette='muted'
)

plt.title('Total Revenue Breakdown by Region & Sales Channel', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Geographical Region', fontsize=12)
plt.ylabel('Total Sales (Rs.)', fontsize=12)

ax = plt.gca()
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: format(int(x), ','))) # keep y-axis values in millions

plt.legend(title='Sales Channel', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = ['price_per_unit', 'units_sold', 'total_sales', 'operating_profit', 'operating_margin']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[idx], color='steelblue')
    axes[idx].set_title(f'Distribution of {col}', fontweight='bold', fontsize=12)
    axes[idx].set_xlabel(col)

    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[idx].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
    axes[idx].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
    axes[idx].legend()

fig.delaxes(axes[5]) # removing extra subplot
plt.tight_layout()
# plt.savefig('distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "-------------------------")
print("Statistical Summary (Using NumPy)")
print("--------------------------")
for col in numerical_cols:
    data = df[col].dropna().values
    print(f"\n{col.upper()}:")
    print(f"  Mean: {np.mean(data):,.2f}")
    print(f"  Median: {np.median(data):,.2f}")
    print(f"  Std Dev: {np.std(data):,.2f}")
    print(f"  Min: {np.min(data):,.2f}")
    print(f"  Max: {np.max(data):,.2f}")
    print(f"  25th Percentile: {np.percentile(data, 25):,.2f}")
    print(f"  75th Percentile: {np.percentile(data, 75):,.2f}")
    print(f"  IQR: {np.percentile(data, 75) - np.percentile(data, 25):,.2f}")
    print(f"  Skewness: {pd.Series(data).skew():.2f}")
    print(f"  Kurtosis: {pd.Series(data).kurtosis():.2f}")